# Create DSL search queries

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Literal

from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [ ]:
POSITION_TITLES = Literal[
    # Consulting & Advisory
    "Associate Consultant",
    "Consultant",
    "Senior Consultant",
    "Managing Consultant",
    "Senior Manager",
    "Principal Consultant",
    "Managing Director",
    "Partner",
    # Data Science & Analytics
    "Junior Data Scientist",
    "Data Scientist",
    "Senior Data Scientist",
    "Staff Data Scientist",
    "Principal Data Scientist",
    "Data Analyst",
    "Senior Data Analyst",
    "Data Engineer",
    "Senior Data Engineer",
    "Machine Learning Engineer",
    "AI Research Scientist",
    # Software & Systems Engineering
    "Associate Software Engineer",
    "Software Engineer",
    "Senior Software Engineer",
    "Staff Software Engineer",
    "Principal Engineer",
    "DevOps Engineer",
    "Cloud Architect",
    "Solutions Architect",
    # Product & Project Management
    "Associate Product Manager",
    "Product Manager",
    "Senior Product Manager",
    "Principal Product Manager",
    "Project Coordinator",
    "Project Manager",
    "Program Manager",
    "Scrum Master",
    # Leadership & Executive
    "Tech Lead",
    "Engineering Manager",
    "Director of Engineering",
    "Director of Data & Analytics",
    "Senior Director",
    "Vice President (VP)",
    "Chief Technology Officer (CTO)",
    "Chief Data Officer (CDO)",
    "Chief Product Officer (CPO)",
    "Chief Executive Officer (CEO)",
    "Chief Operating Officer (COO)",
    "Chief Information Officer (CIO)",
    "Chief Marketing Officer (CMO)",
    "Chief Financial Officer (CFO)",
    "Chief Strategy Officer (CSO)",
    "Chief of Staff (CoS)",
]

In [ ]:
# Coresignal API Standardized Management Levels
MANAGEMENT_LEVELS = Literal[
    "C-Level",
    "Director",
    "Founder",
    "Head",
    "Intern",
    "Manager",
    "Owner",
    "Partner",
    "President/Vice President",
    "Senior",
    "Specialist",
]

# Coresignal API Standardized Departments
DEPARTMENTS = Literal[
    "Administrative",
    "Consulting",
    "Customer Service",
    "Design",
    "Education",
    "Engineering and Technical",
    "Finance & Accounting",
    "General Management",
    "Human Resources",
    "Legal",
    "Marketing",
    "Medical",
    "Operations",
    "Product",
    "Project Management",
    "Real Estate",
    "Research",
    "Sales",
]

In [ ]:
class JobProfile(BaseModel):
    job_location: str = Field(
        description="The location of the job profile, e.g., 'Berlin', 'Remote', 'New York'"
    )
    requirements: list[str] = Field(
        description="Concise list with hard non-negotiable requirements for the job profile"
    )
    nice_to_have: list[str] = Field(
        description="Concise list with nice-to-have requirements for the job profile"
    )
    # years_of_experience: Literal["0-2", "3-5", "6-9", "10+"] = Field(
    #     description="Range of years of experience for the job profile"
    # )
    position_titles: list[POSITION_TITLES] = Field(
        description="Position titles to recruit from",
    )
    management_levels: list[MANAGEMENT_LEVELS] = Field(
        description="List of management levels relevant to the job profile",
    )
    departments: list[DEPARTMENTS] = Field(
        description="List of departments relevant to the job profile",
    )
    keywords: list[str] = Field(
        description="List of keywords relevant to the job profile",
    )

In [ ]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.7-flash",
#     temperature=1.0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )

# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore
# dsl_gen = llm.with_structured_output(
#     DSLQuery,
#     method="function_calling",
#     include_raw=True,
# )

llm = ChatOpenAI(model="gpt-5.6-sol")
structured = llm.with_structured_output(
    JobProfile,
    method="json_schema",
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
system_prompt = f"""
Create a job profile for the job description

## Guidelines
- Requirements and nice-to-have lists must be mutually exclusive and collectively exhaustive.
- List all job titles that suitable candidates could have held / held in the past.
- Job titles and experience levels must match seniority and responsibilities.
- Keywords: Most relevant hard skills like programming languages, frameworks, tools, and methodologies. Do not infer keywords.
""".strip()

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

raw = structured.invoke(messages)
job_profile = JobProfile.model_validate(raw)

In [ ]:
print(json.dumps(job_profile.model_dump(), indent=2))

In [ ]:
stop

## Create DSL Queries

In [ ]:
docs = Path("..") / "docs"
multi_source_dsl_path = (
    docs / "coresignal" / "Employee APIs" / "Multi-source Employee API.json"
)

with open(multi_source_dsl_path, "r") as file:
    multi_source_dsl_json = file.read()

In [ ]:
class DSLQuery(BaseModel):
    query: dict[str, Any] = Field(
        description="Top level query object for the DSL query"
    )

In [ ]:
system_prompt = f"""
Generate a DSL search query in JSON format that finds candidates matching the job description

```json
{multi_source_dsl_json}
```
""".strip()

In [ ]:
# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore
# dsl_gen = llm.with_structured_output(
#     DSLQuery,
#     method="function_calling",
# )

llm = ChatOpenAI(model="gpt-5.6-sol")
dsl_gen = llm.with_structured_output(
    DSLQuery,
    method="json_mode",
)

In [ ]:
prompt_mk = f"""
Job description:
{json.dumps(job_profile.model_dump(), indent=2)}
"""

query_raw = dsl_gen.invoke(
    [
        ("system", system_prompt),
        (
            "human",
            prompt_mk,
        ),
    ]
)
query = DSLQuery.model_validate(query_raw)
print(json.dumps(query.model_dump(), indent=2))